# Study on langchain

Use local ollama service to study langchain

In [2]:
import  os
import langchain
import langchain_ollama as ollama
from langchain_ollama.chat_models import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print(f'Import LangChain V{langchain.__version__}')
print(f'Import Ollma V{ollama.__version__}')

Import LangChain V1.1.3
Import Ollma V1.0.1


## Demo from book

In [11]:
llm = ChatOllama(
    model="deepseek-r1",  # 指定模型，确保已通过 `ollama pull` 下载
    base_url="http://localhost:11434",  # Ollama本地服务地址，默认即此
    temperature=0.7,  # 控制生成随机性
    num_ctx=4096  # 控制上下文长度
)

In [12]:
prompt_extract  =  ChatPromptTemplate.from_template(
  "请从以下文本中提取技术规格：\n\n{text_input}"
)

prompt_transform  =  ChatPromptTemplate.from_template(
  "请将以下技术规格转为 JSON 格式，包含 'CPU'、'内存' 和 '硬盘' 三个键：\n\n{specifications}"
)

extraction_chain = prompt_extract | llm | StrOutputParser()

full_chain = (
    {"specifications":  extraction_chain}
    | prompt_transform | llm | StrOutputParser()
)

In [13]:
input_text = "新款笔记本配备 3.5GHz 八核处理器、16GB 内存和 1TB NVMe SSD。"

final_result = full_chain.invoke({"text_input":  input_text})

print("\n--- 最终 JSON 输出 ---")
print(final_result)


--- 最终 JSON 输出 ---
{
  "CPU": "3.5GHz 八核处理器",
  "内存": "16GB",
  "硬盘": "1TB NVMe SSD"
}


## Try to design another example

* Analysis style and rhetoric from a paragraph
* Summarize into a json

In [19]:
def analysis_chain(model):
    prompt_analysis = ChatPromptTemplate.from_template(
        "Please analysis style and rhetoric from the paragraph: \n\n"
        "{paragraph}"
    )
    prompt_transform = ChatPromptTemplate.from_template(
        "Please transform the analysis result into JSON format, "
        "incluing 'style' and 'rhetoric' keys, using English\n\n"
        "{analysis}"
    )
    analysis = prompt_analysis | model | StrOutputParser()
    return (
        {"analysis": analysis}
        | prompt_transform | model | StrOutputParser()
    )

In [20]:
paragraph = "林野把名字写进风里，把影子交给石砾，把心跳系向那片消失的湖。"
"风蚀的贝壳像白牙，一路啃咬他的鞋底；盐霜像冷笑，一层层揭他的旧疤；黑戈壁像巨兽，一寸寸吞他的水袋。"
"他走，太阳烧他的背；他停，月亮冻他的额；他跪，星群钉他的眼。"
"第七夜，火山口张开黑唇，吐出一圈银白的盐戒。"
"林野伸手，盐粒尖叫着碎成尘；他俯耳，湖魂在泥下低低哭泣。"
"排比的月光：一滴、两滴、千滴，全落在同一道裂缝。"
"排比的喘息：一次、两次、千次，全献给同一口咸水。"
"排比的星：一颗、两颗、千颗，全替他守口如瓶。"
"他把记录仪埋进泥的心脏，让时间做回医生，让盐做回墓碑。"
"回程，沙墙拔地而起，像千万匹脱缰的琥珀马，踏碎他的脚印，嚼烂他的指南针。"
"林野把芯片含在舌下，像含着最后一片冰；他把记忆藏进瞳孔，像藏着最后一颗火；他把名字交给风暴，像交出最后一枚盾。"
"天穹俯身，用星屑缝补他裂开的影子；戈壁侧身，用砾石为他让出一条光的缝隙。"
"他爬，爬得比夜更黑，比盐更涩，比记忆更长。"
"黎明前，他吐出芯片，芯片上只剩一句排比的遗言：我来过，我听见，我带走。"

chain = analysis_chain(llm)

final_result = chain.invoke({"paragraph":  paragraph})

print("\n--- 最终 JSON 输出 ---")
print(final_result)


--- 最终 JSON 输出 ---
```json
{
  "analysis": {
    "style": {
      "poetic_style": true,
      "symbolic_expression": true,
      "rhythmic_pattern": true,
      "concise_language": true,
      "archaic_tone": true,
      "melancholic_atmosphere": true
    },
    "rhetoric": {
      "metaphor": [
        "The name inscribed in the wind",
        "The shadow entrusted to the stone",
        "The heartbeat tethered to the vanished lake"
      ],
      "symbolism": {
        "wind": "forgetfulness, transmission, freedom",
        "stone": "eternity, coldness, finality",
        "lake": "lost beauty, unrequited love, unfulfilled dreams, death"
      },
      "parallelism": [
        "writing the name, entrusting the shadow, tethering the heartbeat",
        "into the wind, to the stone, toward the lake"
      ],
      "repetition": [
        "the word \"into/to/toward\"",
        "the structure \"to verb the object to the recipient\"",
        "the pronoun \"I\" (implied by the verbs)"
   